## Задача 1

Лесник решил провести кластеризацию животных по их расположению в лесу. Он разделил карту на квадраты по километровым отметкам: первый квадрат можно описать 0 ≤ x ≤ 1, 0 ≤ y ≤ 1, второй — 1 ≤ x ≤ 2,0 ≤ y ≤ 1 и так далее.

Для файла А леснику нужно определить два соседних квадрата, в которых суммарно находится больше всего животных. Для файла Б леснику нужно определить три соседних квадрата. Квадраты называются соседними, если у них есть общая граница.

Для каждого файла вычислите два числа: S — количество социальных животных в выбранных соседних квадратах, и K — количество остальных животных в выбранных соседних квадратах. Животное называется социальным, если в радиусе 0.1 вокруг него находится как минимум 14 других животных.

В ответе запишите четыре числа: в первой строке S и K для файла А, во второй строке аналогичные данные для файла Б.

In [59]:
def solve_1(filename, n):
    with open(filename) as f:
        p = [list(map(float, line.replace(',', '.').split())) for line in f]

    gr = {}
    for i in range(len(p)):
        x, y = p[i]
        gx, gy = int(x / 0.1), int(y / 0.1)
        if (gx, gy) not in gr: gr[(gx, gy)] = []
        gr[(gx, gy)].append(i)
    
    soc = [False] * len(p)
    for i in range(len(p)):
        x, y = p[i]
        gx, gy = int(x / 0.1), int(y / 0.1)
        k = 0
        for dx in range(-1, 2):
            for dy in range(-1, 2):
                c = (gx + dx, gy + dy)
                if c in gr:
                    for neighbor_idx in gr[c]:
                        if i == neighbor_idx: continue
                        nx, ny = p[neighbor_idx]
                        if ((x - nx)**2 + (y - ny)**2)**0.5 <= 0.1:
                            k += 1
        if k >= 14:
            soc[i] = True

    sqs = {}
    for i in range(len(p)):
        x, y = p[i]
        sq = (int(x), int(y))
        if sq not in sqs: sqs[sq] = [0, 0]
        if soc[i]:
            sqs[sq][0] += 1
        else:
            sqs[sq][1] += 1
            
    sq_c = list(sqs.keys())
    max_k = -1
    res_s, res_k = 0, 0
    
    def g_neigh(s1):
        return [(s1[0]+1, s1[1]), (s1[0]-1, s1[1]), (s1[0], s1[1]+1), (s1[0], s1[1]-1)]

    if n == 2:
        for s1 in sq_c:
            for s2 in g_neigh(s1):
                if s2 in sqs:
                    s_sum = sqs[s1][0] + sqs[s2][0]
                    k_sum = sqs[s1][1] + sqs[s2][1]
                    k = s_sum + k_sum
                    if k > max_k:
                        max_k, res_s, res_k = k, s_sum, k_sum
                        
    elif n == 3:
        for s1 in sq_c:
            neighs = [n for n in g_neigh(s1) if n in sqs]
            for i in range(len(neighs)):
                for j in range(i + 1, len(neighs)):
                    s2, s3 = neighs[i], neighs[j]
                    s_sum = sqs[s1][0] + sqs[s2][0] + sqs[s3][0]
                    k_sum = sqs[s1][1] + sqs[s2][1] + sqs[s3][1]
                    k = s_sum + k_sum
                    if k > max_k:
                        max_k, res_s, res_k = k, s_sum, k_sum
                
                s2 = neighs[i]
                for s3 in g_neigh(s2):
                    if s3 in sqs and s3 != s1:
                        s_sum = sqs[s1][0] + sqs[s2][0] + sqs[s3][0]
                        k_sum = sqs[s1][1] + sqs[s2][1] + sqs[s3][1]
                        k = s_sum + k_sum
                        if k > max_k:
                            max_k, res_s, res_k = k, s_sum, k_sum

    return res_s, res_k

In [60]:
print(*solve_1('1_A.txt', 2))
print(*solve_1('1_B.txt', 3))

104 453
2156 158


## Задача 2

Научно﻿-﻿исследовательский институт проводит мониторинг экологического состояния различных регионов. Результаты измерений представляются в виде пары чисел: первое — концентрация загрязняющего вещества в почве, второе — концентрация того же вещества в близлежащем водоёме. Для анализа результатов эта пара рассматривается как координаты точки на плоскости, и строится график с точками, которые соответствуют всем измерениям.

По ошибке данные нескольких исследуемых регионов были записаны в один файл. Известно, что измерения относятся к одному региону, если они образуют компактные группы точек на графике. Каждая группа лежит внутри прямоугольника высотой H и шириной.

Перед проведением основного анализа необходимо очистить данные от случайных выбросов при измерении. Для этого используется метод межквартильного размаха:
* для каждой группы точек вычисляется первый квартиль Q1 (значение, ниже которого находится 25% измерений) и третий квартиль Q3 (значение, выше которого находится 25% измерений) отдельно для рядов значений концентрации загрязняющего вещества в почве и в близлежащем водоёме (координаты X и Y)
* вычисляется межквартильный размах IQR - Q3 - Q1 для каждой кординаты
* точка считается выбросом, если хотя бы одна из её координат Х или Y выходит за пределы диапазона |Q1 - 1.5 * IQR; Q3 + 1.5 * IQR|

Для каждого региона необходимо рассчитать индекс экологической опасности I, который определяется как отношение среднего значения измерений к их количеству после удаления выбросов.

Под средним значением измерений в этом случае понимается среднее евклидово расстояние между всеми парами различных точек (измерений) в регионе.

В файле A хранятся данные о трёх регионах, где H=30, W=35. Каждая строка файла содержит два числа: координаты X (концентрация в почве) и Y (концентрация в водоёме), соответствующие одному измерению. Значения даны в условных единицах. Известно, что количество измерений не превышает 1000.

В файле B хранятся данные о пяти регионах, где H=40, W=32. Известно, что количество измерений не превышает 10000. Структура хранения информации в файле B аналогична файлу А.

Для каждого файла определите:
* общее количество выявленных выбросов
* регион с максимальным индексом экологической опасности

В ответе запишите четыре числа: в первой строке — общее количество выбросов и целую часть произведения I×100000 для файла A, во второй строке — аналогичные данные для файла B.

Расчёт квартилей
* Найдите медиану значений данных. Это второй квартиль Q2
* Найдите медиану значений данных, которые находятся ниже второго квартиля. Это первый квартиль Q1
* Найдите медиану значений данных, которые выше второго квартиля. Это третий квартиль Q3

In [62]:
import math

def med(ms):
    n = len(ms)
    if n % 2 != 0:
        return ms[n // 2]
    return (ms[n // 2 - 1] + ms[n // 2]) / 2

def quar(arr):
    s = sorted(arr)
    n = len(s)
    mid = n // 2
    q1 = med(s[:mid])
    q3 = med(s[mid + (1 if n % 2 != 0 else 0):])
    return q1, q3

def solve_2(filename, H, W):
    with open(filename) as f:
        p = [list(map(float, line.replace(',', '.').split())) for line in f]

    cls = []
    us = [False] * len(p)
    for i in range(len(p)):
        if not us[i]:
            cl = []
            stack = [i]
            us[i] = True
            while stack:
                curr_idx = stack.pop()
                curr_pt = p[curr_idx]
                cl.append(curr_pt)
                for j in range(len(p)):
                    if not us[j]:
                        if abs(curr_pt[0] - p[j][0]) <= W and \
                           abs(curr_pt[1] - p[j][1]) <= H:
                            us[j] = True
                            stack.append(j)
            cls.append(cl)

    k = 0
    mx = 0

    for cl in cls:
        xs = [p[0] for p in cl]
        ys = [p[1] for p in cl]
        
        q1x, q3x = quar(xs)
        q1y, q3y = quar(ys)
        
        iqr_x, iqr_y = q3x - q1x, q3y - q1y
        
        low_x, upp_x = q1x - 1.5 * iqr_x, q3x + 1.5 * iqr_x
        low_y, upp_y = q1y - 1.5 * iqr_y, q3y + 1.5 * iqr_y
        
        clean = []
        for p in cl:
            if low_x <= p[0] <= upp_x and low_y <= p[1] <= upp_y:
                clean.append(p)
            else:
                k += 1
        
        n = len(clean)
        if n > 1:
            sum = 0
            for i in range(n):
                for j in range(i + 1, n):
                    d = math.sqrt((clean[i][0] - clean[j][0])**2 + 
                                  (clean[i][1] - clean[j][1])**2)
                    sum += d
            
            avg_dist = sum / (n * (n - 1) / 2)
            mx = max(mx, avg_dist / n)

    return k, int(mx * 100000)

In [63]:
print(*solve_2('2_A.txt', 30, 35))
print(*solve_2('2_B.txt', 40, 32))

25 3804
467 189


## Задача 3

Кластеризуйте как в предыдущих домашках. Будем называть центром кластера точку этого кластера, сумма расстояний от которой до всех остальных точек кластера минимальна.

Для каждого файла определите координаты центра каждого кластера, затем вычислите два числа: A - среднее арифметическое абсцисс центров кластеров, и среднее B - арифметическое ординат центров кластеров. В ответе запишите четыре числа: в первой строке сначала целую часть произведения A\*10000 затем целую часть произведения B\*10000 для файла А, во второй строке — аналогичные данные для файла Б.


In [50]:
def dist(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def solve_3(filename, eps):
    p = []
    with open(filename) as f:
        for line in f:
            if line.strip():
                p.append(list(map(float, line.replace(',', '.').split())))

    cls= []
    us = [False] * len(p)
    
    for i in range(len(p)):
        if not us[i]:
            cl = []
            st = [i]
            us[i] = True
            while st:
                curr_idx = st.pop()
                curr_p = p[curr_idx]
                cl.append(curr_p)
                for j in range(len(p)):
                    if not us[j]:
                        if dist(curr_p, p[j]) < eps:
                            us[j] = True
                            st.append(j)
            cls.append(cl)

    c_x = []
    c_y = []
    
    for cl in cls:
        min_dist = float('inf')
        b_p = cl[0]
        
        for p1 in cl:
            c_sum = 0
            for p2 in cl:
                c_sum += dist(p1, p2)
            
            if c_sum < min_dist:
                min_dist = c_sum
                b_p = p1
        
        c_x.append(b_p[0])
        c_y.append(b_p[1])

    m_x = sum(c_x) / len(c_x)
    m_y = sum(c_y) / len(c_y)
    
    return int(m_x * 10000), int(m_y * 10000)


In [49]:
print(*solve_3('3_A.txt', eps=5))
print(*solve_3('3_B.txt', eps=5))

254624 48396
-52968 63812


## Задача 4

Исследователь анализирует набор объектов, каждый из которых характеризуется пятью числовыми параметрами. Он знает, что объекты образуют несколько групп (кластеров), которые можно выявить при проекции на плоскость только двух параметров из пяти. Значения в одном из столбцов будут соответствовать координатам по оси абсцисс, а из второго — координатам по оси ординат. Каждый кластер можно заключить в квадратную область заданного размера L, причём эти квадраты между собой не пересекаются. Стороны квадратов параллельны координатным осям. Каждый объект должен принадлежать только одному кластеру.

Евклидово расстояние

В файле A хранятся данные о наборе объектов, образующих три кластера. В каждой строке через пробел записаны пять параметров, характеризующих один объект. Все значения представлены с точностью до двух знаков после запятой. Количество объектов в файле А не превышает 1000.

В файле Б записаны данные о наборе объектов, образующих шесть кластеров, с аналогичной структурой хранения информации. Количество объектов в файле Б не превышает 10000.

Для каждого файла необходимо определить, какая пара параметров позволяет разделить объекты на кластеры, и найти минимальный размер стороны квадрата L, который может содержать все точки одного кластера при проекции на плоскость найденных параметров. Также определите в каждом кластере расстояние между двумя объектами, расположенными дальше всего друг от друга, и вычислите P — среднее арифметическое таких расстояний для всех кластеров.

В ответе запишите четыре числа: в первой строке — целую часть произведения L×10000, затем целую часть произведения P×10000 для файла А, во второй строке — аналогичные значения для файла Б.

In [66]:
def get_dist(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def get_cls(p, L):
    n = len(p)
    us = [False] * n
    cls = []
    for i in range(n):
        if not us[i]:
            cl = []
            stack = [i]
            us[i] = True
            while stack:
                curr = stack.pop()
                cl.append(p[curr])
                for j in range(n):
                    if not us[j]:
                        if abs(p[curr][0] - p[j][0]) <= L and \
                           abs(p[curr][1] - p[j][1]) <= L:
                            us[j] = True
                            stack.append(j)
            cls.append(cl)
    return cls

def solve_4(filename, n):
    ms = []
    with open(filename) as f:
        for line in f:
            if line.strip():
                ms.append(list(map(float, line.replace(',', '.').split())))

    best_L = 0
    best_P = 0
    
    for c1 in range(5):
        for c2 in range(c1 + 1, 5):
            proj = [[row[c1], row[c2]] for row in ms]
            L = 0.05
            f_cls = []
            while L < 10.0:
                cls = get_cls(proj, L)
                if len(cls) == n:
                    f_cls = cls
                    break
                L += 0.05
            
            if len(f_cls) == n:
                max_L_in_cls = 0
                for cl in f_cls:
                    xs = [p[0] for p in cl]
                    ys = [p[1] for p in cl]
                    side = max(max(xs) - min(xs), max(ys) - min(ys))
                    max_L_in_cls = max(max_L_in_cls, side)

                sm = 0
                for cl in f_cls:
                    max_d = 0
                    for i in range(len(cl)):
                        for j in range(i + 1, len(cl)):
                            d = get_dist(cl[i], cl[j])
                            if d > max_d: max_d = d
                    sm += max_d
                
                avg_P = sm / n

                best_L = max_L_in_cls
                best_P = avg_P
                break
                
    return int(best_L * 10000), int(best_P * 10000)


In [67]:
print(*solve_4('4_A.txt', 3))
print(*solve_4('4_B.txt', 6))

992400 950249
999600 666758


In [33]:
## Задача 5

In [68]:
def get_dist(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def get_cls(p, L):
    n = len(p)
    us = [False] * n
    cls = []
    
    for i in range(n):
        if not us[i]:
            cl = []
            stack = [i]
            us[i] = True
            while stack:
                curr_idx = stack.pop()
                curr_pt = p[curr_idx]
                cl.append(curr_pt)
                for j in range(n):
                    if not us[j]:
                        if abs(curr_pt[0] - p[j][0]) <= L and \
                           abs(curr_pt[1] - p[j][1]) <= L:
                            us[j] = True
                            stack.append(j)
            cls.append(cl)
    return cls

def solve_5(filename, n):
    ms = []
    with open(filename, 'r') as f:
        for line in f:
            if line.strip():
                ms.append(list(map(float, line.replace(',', '.').split())))

    num_cols = len(ms[0])
    best_L, best_P = 0.0, 0.0

    for c1 in range(num_cols):
        for c2 in range(c1 + 1, num_cols):
            proj = [[row[c1], row[c2]] for row in ms]
            
            for L_step in [x * 0.1 for x in range(1, 100)]: 
                cls = get_cls(proj, L_step)
                
                if len(cls) == n:
                    mx_s = 0
                    for cl in cls:
                        xs = [p[0] for p in cl]
                        ys = [p[1] for p in cl]
                        s = max(max(xs) - min(xs), max(ys) - min(ys))
                        if s > mx_s:
                            mx_s = s
                    
                    mx_dist = 0
                    for cl in cls:
                        max_d = 0
                        for i in range(len(cl)):
                            for j in range(i + 1, len(cl)):
                                d = get_dist(cl[i], cl[j])
                                if d > max_d: max_d = d
                        mx_dist += max_d
                    
                    avg_P = mx_dist / n
                    return mx_s, avg_P
                    
    return 0, 0

a = solve_5('5_A.txt', 3)
b = solve_5('5_B.txt', 5)

print(f"{int(a[0] * 10000)} {int(a[1] * 10000)}")
print(f"{int(b[0] * 10000)} {int(b[1] * 10000)}")


19056 18049
28437 20305
